Detector e contador simples de veículos por subtração de backgrounds

In [ ]:
import cv2
import numpy as np
import math


cap = cv2.VideoCapture("highway.mp4")


object_detector = cv2.createBackgroundSubtractorMOG2(
    history=100, varThreshold=40, detectShadows=False)

# ROI no frame original
X1, Y1 = 500, 340
X2, Y2 = 800, 720

# Linha de contagem, em coordenada Y relativa à ROI
linha_contagem = 200

distancia_maxima = 50
max_frames_perdido = 10

objetos = {}
id_global = 0
total_veiculos = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # A detecção é executada somente nesta região
    roi = frame[Y1:Y2, X1:X2]
    mask = object_detector.apply(roi)

    _, mask = cv2.threshold(mask, 254, 255, cv2.THRESH_BINARY)

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.dilate(mask, kernel, iterations=2)

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    deteccoes = []
    for cnt in contours:
        if cv2.contourArea(cnt) > 100:
            x, y, w, h = cv2.boundingRect(cnt)
            cx, cy = x + w // 2, y + h // 2
            deteccoes.append((cx, cy, x, y, w, h))

    for obj in objetos.values():
        obj["atualizado"] = False

    # Possíveis associações: distância entre centroides
    pares = []
    for id_obj, obj in objetos.items():
        for i, det in enumerate(deteccoes):
            cx, cy = det[0], det[1]
            distancia = math.hypot(cx - obj["cx"], cy - obj["cy"])
            if distancia < distancia_maxima:
                pares.append((distancia, id_obj, i))

    pares.sort()
    ids_usados = set()
    deteccoes_usadas = set()

    for distancia, id_obj, i in pares:
        if id_obj in ids_usados or i in deteccoes_usadas:
            continue

        cx, cy, x, y, w, h = deteccoes[i]
        obj = objetos[id_obj]
        cy_anterior = obj["cy"]

        obj.update({
            "cx": cx, "cy": cy, "bbox": (x, y, w, h),
            "perdido": 0, "atualizado": True
        })

        if not obj["contado"]:
            cruzou = (
                cy_anterior < linha_contagem <= cy
                or cy_anterior > linha_contagem >= cy
            )
            if cruzou:
                total_veiculos += 1
                obj["contado"] = True
                print(f"ID {id_obj} contabilizado. Total: {total_veiculos}")

        obj["cy_anterior"] = cy
        ids_usados.add(id_obj)
        deteccoes_usadas.add(i)

    # Novas detecções recebem novos IDs
    for i, det in enumerate(deteccoes):
        if i in deteccoes_usadas:
            continue
        cx, cy, x, y, w, h = det
        id_global += 1
        objetos[id_global] = {
            "cx": cx, "cy": cy, "cy_anterior": cy,
            "bbox": (x, y, w, h), "perdido": 0,
            "atualizado": True, "contado": False
        }

    # Remover rastros perdidos
    for id_obj in list(objetos):
        if not objetos[id_obj]["atualizado"]:
            objetos[id_obj]["perdido"] += 1
            if objetos[id_obj]["perdido"] > max_frames_perdido:
                del objetos[id_obj]

    # Desenhar bounding boxes e IDs no frame completo
    for id_obj, obj in objetos.items():
        if obj["perdido"] > 0:
            continue

        x, y, w, h = obj["bbox"]
        X, Y = x + X1, y + Y1

        cv2.rectangle(frame, (X, Y), (X + w, Y + h), (0, 255, 0), 2)
        cv2.putText(
            frame, f"ID {id_obj}", (X, max(Y - 8, 20)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2
        )
        cv2.circle(frame, (X + w // 2, Y + h // 2), 4, (0, 0, 255), -1)

    # Delimitar ROI e linha virtual no frame completo
    cv2.rectangle(frame, (X1, Y1), (X2, Y2), (255, 0, 0), 2)
    linha_y_frame = Y1 + linha_contagem
    cv2.line(frame, (X1, linha_y_frame), (X2, linha_y_frame), (0, 255, 255), 2)

    cv2.putText(frame, f"Total de veiculos: {total_veiculos}", (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

    cv2.imshow("Detector e Contador - Frame completo", frame)
    cv2.imshow("Mascara ROI", mask)

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()

cv2.destroyAllWindows()
print(f"Total final de veiculos: {total_veiculos}")